# C1 · PCE Estimator (L3 互信息下界)

**Track C · Week 1–2 · Colab Pro L4 22GB(副号)**

## 设计选择
- **W1**:纯 PyTorch PCE 估计器,闭合式 toy(K-component GMM)验证误差 <10%
- **W1 末**:mock PyBaMM forward(解析等效电路 + 高斯噪声)→ 端到端 pipeline 通
- **W2**:真 PyBaMM swap,Hallemans-style 特征抽取(20D)+ 特征空间 Gaussian likelihood
- **高维 y 处理**:200-freq EIS 谱 → 20D 特征,特征空间 Gaussian 似然可解析

## PCE 公式
```
I(H; Y | D) ≥ E_{h_i ~ p(h), y_i ~ p(y|h_i,D)}
              [ log p(y_i|h_i,D) − log (1/K) Σ_k p(y_i|h_k,D) ]
```
(Foster 2020 NeurIPS · h_k 为对比假设池,first-column 约定 h_k[:,0]=h_i)

In [ ]:
# ========== 0. setup ==========
!nvidia-smi | head -5
from google.colab import drive
drive.mount('/content/drive')

import os, json, math
from pathlib import Path
import numpy as np, torch
import torch.nn.functional as F

PHYRE_ROOT = Path('/content/drive/MyDrive/phyre')
SRC_OUT    = PHYRE_ROOT / 'src' / 'pce_estimator.py'
TEST_OUT   = PHYRE_ROOT / 'src' / 'test_pce.py'
for p in [SRC_OUT.parent]: p.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
print('device:', DEVICE)

In [ ]:
# ========== 1. core PCE estimator ==========
def pce_mi_lower_bound(log_p_y_given_h, log_p_y_given_hk):
    """Prior-Contrastive Estimation of I(H; Y).

    Args:
      log_p_y_given_h:  (N,)    log p(y_i | h_i, D)  — must use each sample's real h_i
      log_p_y_given_hk: (N, K)  log p(y_i | h_k, D)  for the contrast pool

    Returns:
      scalar tensor: MI lower bound in nats
    """
    N, K = log_p_y_given_hk.shape
    log_mean = torch.logsumexp(log_p_y_given_hk, dim=1) - math.log(K)
    return (log_p_y_given_h - log_mean).mean()


def identifiability_gap(posterior_probs, tau_ret=0.3, tau_def=0.8):
    """Normalised entropy ∈ [0,1]: 0 = sharp (return), 1 = uniform (defer).

    Returns:  (gap, action) where action ∈ {'return', 'return_with_alt', 'defer'}

    Semantics:
      gap < tau_ret  → return              (posterior sharp enough to answer)
      gap < tau_def  → return_with_alt     (borderline, answer + list alternatives)
      gap ≥ tau_def  → defer               (cannot distinguish, hand off to L4)
    """
    p = torch.as_tensor(posterior_probs, dtype=torch.float32).clamp(min=1e-12)
    p = p / p.sum()
    K = len(p)
    H = -(p * p.log()).sum().item()
    gap = H / math.log(K) if K > 1 else 0.0
    if gap < tau_ret: action = 'return'
    elif gap < tau_def: action = 'return_with_alt'
    else: action = 'defer'
    return gap, action

In [ ]:
# ========== 2. toy benchmark — closed-form MI on K-component GMM ==========
# H uniform over K classes; Y | H=k ~ N(mu_k, sigma^2 I_d)
# Closed form:  I(H;Y) = H(Y) − H(Y|H),  H(Y|H) = d/2 log(2 pi e sigma^2)
# H(Y) via Monte-Carlo on mixture density

def gaussian_logpdf(y, mu, sigma2):
    d = y.shape[-1]
    return -0.5 * ((y - mu) ** 2).sum(-1) / sigma2 - 0.5 * d * math.log(2 * math.pi * sigma2)

def toy_true_mi(mus, sigma2, n_mc=20000):
    K, d = mus.shape
    h_idx = torch.randint(0, K, (n_mc,))
    y = mus[h_idx] + torch.randn(n_mc, d) * math.sqrt(sigma2)
    log_pyk = torch.stack([gaussian_logpdf(y, mus[k], sigma2) for k in range(K)], dim=1)
    log_py = torch.logsumexp(log_pyk, dim=1) - math.log(K)
    H_y = -log_py.mean()
    H_y_given_h = 0.5 * d * math.log(2 * math.pi * math.e * sigma2)
    return (H_y - H_y_given_h).item()

def toy_pce_estimate(mus, sigma2, N=10000):
    K, d = mus.shape
    h_idx = torch.randint(0, K, (N,))
    y = mus[h_idx] + torch.randn(N, d) * math.sqrt(sigma2)
    # FIX: log p(y_i|h_i) must use each sample's own mu (not zeros)
    log_p_y_given_h  = gaussian_logpdf(y - mus[h_idx], torch.zeros(d), sigma2)   # (N,)
    log_p_y_given_hk = torch.stack([gaussian_logpdf(y, mus[k], sigma2)
                                     for k in range(K)], dim=1)                 # (N, K)
    return pce_mi_lower_bound(log_p_y_given_h, log_p_y_given_hk).item()

# run toy
print(f'{"config":30s} {"true MI":>10s} {"PCE lb":>10s} {"err %":>8s}  pass')
for K, d, sigma2 in [(4, 2, 1.0), (4, 5, 1.0), (8, 2, 0.5), (4, 10, 2.0)]:
    torch.manual_seed(1)
    mus = torch.randn(K, d) * 2
    true_mi = toy_true_mi(mus, sigma2)
    pce_mi  = toy_pce_estimate(mus, sigma2, N=10000)
    err = abs(pce_mi - true_mi) / max(abs(true_mi), 1e-3)
    flag = '✓' if err < 0.10 else '✗'
    cfg = f'K={K} d={d} σ²={sigma2}'
    print(f'{cfg:30s} {true_mi:>10.3f} {pce_mi:>10.3f} {err*100:>7.1f}%   {flag}')

In [ ]:
# ========== 3. mock PyBaMM forward (analytic Randles + noise) — W1 使用,W2 swap 真 PyBaMM ==========
# Randles impedance:  Z(ω) = R_s + R_ct / (1 + j ω R_ct C_dl) + σ/√(jω)   (Warburg)
# Hypothesis h = (R_s, R_ct, C_dl, sigma_W)  — 4-dim parameter vector

FREQS = torch.logspace(-2, 5, 200)  # 0.01 Hz → 100 kHz

def randles_impedance(params, freqs=FREQS):
    """params: (B, 4) [R_s, R_ct, C_dl, sigma_W].  Returns (B, 200, 2) [Re, -Im].
    
    Shape discipline:
      params  (B, 4) → unbind → each (B,) → unsqueeze(-1) → (B, 1)
      freqs   (F,)   →        → jw       → (F,) complex
      broadcast (B, 1) * (F,) → (B, F)
    """
    R_s, R_ct, C_dl, sig = params.unbind(-1)                                 # each (B,)
    R_s  = R_s.unsqueeze(-1);  R_ct = R_ct.unsqueeze(-1)
    C_dl = C_dl.unsqueeze(-1); sig  = sig.unsqueeze(-1)                      # (B, 1)
    omega = 2 * math.pi * freqs                                              # (F,)
    jw = 1j * omega.to(torch.complex64)                                      # (F,) complex64
    Z_ct = R_ct / (1 + jw * R_ct * C_dl)                                     # (B, F)
    Z_w  = sig / torch.sqrt(jw)                                              # (B, F) via broadcast
    Z = R_s + Z_ct + Z_w                                                     # (B, F)
    return torch.stack([Z.real, -Z.imag], dim=-1).float()                    # (B, F, 2)

def extract_features(spec):
    """20D Hallemans-style features from (B, 200, 2)."""
    re, im = spec[..., 0], spec[..., 1]                                      # (B, 200)
    mag = torch.sqrt(re**2 + im**2)
    re_bins  = re.reshape(*re.shape[:-1], 10, 20).mean(-1)                   # (B, 10)
    mag_bins = mag.reshape(*mag.shape[:-1], 5, 40).mean(-1)                  # (B, 5)
    feats = torch.cat([
        re_bins, mag_bins,
        re.min(-1).values.unsqueeze(-1), re.max(-1).values.unsqueeze(-1),
        im.min(-1).values.unsqueeze(-1), im.max(-1).values.unsqueeze(-1),
        mag.mean(-1).unsqueeze(-1)
    ], dim=-1)                                                                # (B, 20)
    return feats

# 4 hypotheses — physically distinct parameter sets
HYP = torch.tensor([
    [ 5.0, 15.0, 1e-4, 2.0],   # h1: standard
    [ 5.0, 30.0, 1e-4, 2.0],   # h2: 2x R_ct — slow kinetics
    [ 5.0, 15.0, 5e-5, 2.0],   # h3: half C_dl — thinner DL
    [ 8.0, 15.0, 1e-4, 5.0],   # h4: high Warburg — diffusion dominated
])
NOISE_STD = 0.3  # feature-space noise σ

def simulate(h_idx, n=32):
    """Sample n noisy feature vectors conditional on hypothesis h_idx."""
    params = HYP[h_idx].unsqueeze(0).expand(n, -1).contiguous()              # (n, 4)
    spec = randles_impedance(params)                                         # (n, 200, 2)
    feat = extract_features(spec)                                            # (n, 20)
    return feat + torch.randn_like(feat) * NOISE_STD

# sanity check — shapes + per-h feature means
for k in range(4):
    f = simulate(k, 32)
    assert f.shape == (32, 20), f'bad shape {f.shape}'
    print(f'  h{k+1}  feat[0:5] mean = {f.mean(0)[:5].numpy()}')

In [ ]:
# ========== 4. feature-space Gaussian likelihood → PCE ==========
# For each hypothesis h_k, compute empirical mean/cov in feature space from simulator.
# Then log p(y | h_k) = log N(y; μ_k, Σ_k) (diagonal for stability; full cov optional).

def fit_gaussian_per_h(K=4, n_fit=256):
    mus, log_vars = [], []
    for k in range(K):
        f = simulate(k, n_fit)
        mus.append(f.mean(0))
        log_vars.append((f.var(0) + 1e-4).log())
    return torch.stack(mus), torch.stack(log_vars)  # (K, 20), (K, 20)

def diag_gauss_logpdf(y, mu, log_var):
    # y: (N, 20),  mu: (20,),  log_var: (20,)
    return (-0.5 * ((y - mu)**2 / log_var.exp()).sum(-1)
            - 0.5 * log_var.sum()
            - 0.5 * y.shape[-1] * math.log(2 * math.pi))

mus, log_vars = fit_gaussian_per_h(K=4, n_fit=512)
print('fitted per-h Gaussians;  mu shape', mus.shape)

# PCE estimate: sample h uniform, y ~ simulator(h), compute log-ratio
N, K = 2000, 4
h_idx = torch.randint(0, K, (N,))
# one y per h_i
y = torch.stack([simulate(int(h.item()), 1).squeeze(0) for h in h_idx])   # (N, 20)

log_p_y_given_h  = torch.stack([diag_gauss_logpdf(y[i:i+1], mus[h_idx[i]], log_vars[h_idx[i]])
                                for i in range(N)]).squeeze()             # (N,)
log_p_y_given_hk = torch.stack([diag_gauss_logpdf(y, mus[k], log_vars[k])
                                for k in range(K)], dim=1)                # (N, K)

mi_lb = pce_mi_lower_bound(log_p_y_given_h, log_p_y_given_hk)
print(f'\n  PCE MI lower bound (mock Randles, K=4, N=2000):  {mi_lb.item():.3f} nats')
print(f'  upper bound log K = {math.log(K):.3f};  fraction of log K = {mi_lb.item()/math.log(K)*100:.1f}%')

In [ ]:
# ========== 5. pairwise KL spread + merge filter ==========
def sym_kl_gaussian(mu1, lv1, mu2, lv2):
    v1, v2 = lv1.exp(), lv2.exp()
    kl_12 = 0.5 * ((v1 / v2).sum() + ((mu2 - mu1)**2 / v2).sum() - len(mu1) + (lv2 - lv1).sum())
    kl_21 = 0.5 * ((v2 / v1).sum() + ((mu1 - mu2)**2 / v1).sum() - len(mu1) + (lv1 - lv2).sum())
    return 0.5 * (kl_12 + kl_21).item()

K = mus.size(0)
KL = np.zeros((K, K))
for i in range(K):
    for j in range(K):
        if i != j: KL[i, j] = sym_kl_gaussian(mus[i], log_vars[i], mus[j], log_vars[j])
print('pairwise symmetric KL matrix:')
print(np.round(KL, 2))

# merge filter: if any pair < tau_merge, collapse them
TAU_MERGE = 0.5  # nats — tune on benchmark in W5
merge_pairs = [(i, j) for i in range(K) for j in range(i+1, K) if KL[i, j] < TAU_MERGE]
print(f'\npairs to merge (KL < {TAU_MERGE}): {merge_pairs}')

In [ ]:
# ========== 6. IdentGap demo ==========
# simulate posterior probabilities over K hypotheses at different certainty levels
for probs in [[0.95, 0.02, 0.02, 0.01],    # sharp — should return
              [0.7, 0.2, 0.05, 0.05],      # fairly sharp — return
              [0.5, 0.4, 0.05, 0.05],      # borderline — return_with_alt
              [0.3, 0.3, 0.2, 0.2]]:       # flat — should defer
    gap, action = identifiability_gap(probs)
    print(f'  p={probs}  IdentGap={gap:.3f}  action={action}')

In [ ]:
# ========== 7. export to src/pce_estimator.py + src/test_pce.py ==========
MODULE = '"""Prior-Contrastive Estimation + IdentGap (L3 core)."""\nfrom __future__ import annotations\nimport math\nimport torch\n\n\ndef pce_mi_lower_bound(log_p_y_given_h: torch.Tensor,\n                       log_p_y_given_hk: torch.Tensor) -> torch.Tensor:\n    N, K = log_p_y_given_hk.shape\n    log_mean = torch.logsumexp(log_p_y_given_hk, dim=1) - math.log(K)\n    return (log_p_y_given_h - log_mean).mean()\n\n\ndef diag_gauss_logpdf(y: torch.Tensor, mu: torch.Tensor,\n                      log_var: torch.Tensor) -> torch.Tensor:\n    return (-0.5 * ((y - mu) ** 2 / log_var.exp()).sum(-1)\n            - 0.5 * log_var.sum()\n            - 0.5 * y.shape[-1] * math.log(2 * math.pi))\n\n\ndef sym_kl_gaussian(mu1, lv1, mu2, lv2) -> float:\n    v1, v2 = lv1.exp(), lv2.exp()\n    kl_12 = 0.5 * ((v1 / v2).sum() + ((mu2 - mu1) ** 2 / v2).sum()\n                    - len(mu1) + (lv2 - lv1).sum())\n    kl_21 = 0.5 * ((v2 / v1).sum() + ((mu1 - mu2) ** 2 / v1).sum()\n                    - len(mu1) + (lv1 - lv2).sum())\n    return 0.5 * (kl_12 + kl_21).item()\n\n\ndef identifiability_gap(posterior_probs, tau_ret=0.3, tau_def=0.8):\n    """Normalised entropy in [0,1]: 0 = sharp (return), 1 = uniform (defer)."""\n    p = torch.as_tensor(posterior_probs, dtype=torch.float32).clamp(min=1e-12)\n    p = p / p.sum()\n    K = len(p)\n    H = -(p * p.log()).sum().item()\n    gap = H / math.log(K) if K > 1 else 0.0\n    if gap < tau_ret: action = "return"\n    elif gap < tau_def: action = "return_with_alt"\n    else: action = "defer"\n    return gap, action\n'

TEST = '"""Unit tests for pce_estimator."""\nimport math, torch\nfrom pce_estimator import (pce_mi_lower_bound, identifiability_gap,\n                            diag_gauss_logpdf, sym_kl_gaussian)\n\ndef test_pce_bounded_above_log_K():\n    torch.manual_seed(0); N, K = 1000, 4\n    log_p = torch.zeros(N); log_pk = torch.zeros(N, K)\n    mi = pce_mi_lower_bound(log_p, log_pk)\n    assert mi.item() <= math.log(K) + 1e-4\n\ndef test_pce_near_log_K_when_well_separated():\n    torch.manual_seed(0); N, K, d = 4000, 4, 5\n    mus = torch.eye(K, d) * 10\n    h = torch.randint(0, K, (N,))\n    y = mus[h] + torch.randn(N, d) * 0.1\n    lv = torch.zeros(d) - 4\n    lpy_h = torch.stack([diag_gauss_logpdf(y[i:i+1], mus[h[i]], lv) for i in range(N)]).squeeze()\n    lpy_hk = torch.stack([diag_gauss_logpdf(y, mus[k], lv) for k in range(K)], 1)\n    mi = pce_mi_lower_bound(lpy_h, lpy_hk)\n    assert mi.item() > 0.9 * math.log(K)\n\ndef test_identgap_return_on_sharp():\n    g, a = identifiability_gap([0.95, 0.02, 0.02, 0.01])\n    assert a == "return", f"got {a} gap={g}"\n\ndef test_identgap_defer_on_flat():\n    g, a = identifiability_gap([0.3, 0.3, 0.2, 0.2])\n    assert a == "defer", f"got {a} gap={g}"\n\ndef test_identgap_borderline():\n    g, a = identifiability_gap([0.5, 0.4, 0.05, 0.05])\n    assert a == "return_with_alt", f"got {a} gap={g}"\n\ndef test_sym_kl_zero_for_identical():\n    mu = torch.zeros(5); lv = torch.zeros(5)\n    assert abs(sym_kl_gaussian(mu, lv, mu, lv)) < 1e-5\n'

SRC_OUT.write_text(MODULE, encoding='utf-8')
TEST_OUT.write_text(TEST, encoding='utf-8')
print(f'wrote {SRC_OUT}')
print(f'wrote {TEST_OUT}')
print('\nto run tests locally:  cd ~/phyre/src && pytest test_pce.py -v')